# Capstone — mirrors your deployed research paper

This notebook mirrors the deployed paper, live at:
https://fatima-eman-hub.github.io/fatima-eman-flyrank-ml-01/

> Working with an AI assistant? Tell it to read `skills/README.md` first and load
> `writing-research-papers` + `deploying-static-pages` for this task.

## 1. Question

*The research question and the decision it supports.*

**Research question:** Given everything knowable about a content page today,
which pages are worth a human reviewer's limited time first?

**Decision this supports:** FlyRank content teams review large inventories
of client pages with limited time. The existing hand-written rule (stale AND
visible) is a coarse filter that cannot weigh position, traffic, and
freshness together. This work replaces that filter with a validated,
client-grouped model that ranks pages for manual review — not an automatic
action-taker. Lane: Refresh / Content Opportunity Scoring (Lane 2).

## 2. Data

*Which release, tables, date windows, what you excluded and why. Public-safe.*

**Release:** FlyRank ML Internship warehouse, build `v20260703` (Hugging Face,
gated, instant approval).

**Tables used:**
- `fact_content_daily_performance` — 78,835,655 rows, grain of
  report_date × client × content item, 2025-01-27 to 2026-06-30.
- `dim_content` — 519,606 rows, one per pseudonymized content item.

**Date windows:** features from March 2026 (mid-panel month), label from
April 2026 (future window) — so every feature is knowable strictly before
the outcome it predicts.

**Excluded, and why:** FlyRank's own product decision flags (`health_score`,
`priority_score`, `action_type`, `refresh_tier`) are never used — including
them would let the model re-learn the existing rule instead of finding new
signal. No raw client names, domains, URLs, or query text appear anywhere
in this work.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Assumption:** a page's near-term trajectory is predictable from its own
recent observable signals, and this relationship is reasonably stable across
the ~70 clients in this panel (history depth varies — see Limitations).

**Features** (all from March 2026 or a fixed past date, never the label window):
- `impressions_march` — knowable because GSC impressions log as they happen.
- `avg_position` — recorded daily, fully observed by month-end.
- `content_age_days` — derived from a fixed past creation date.
- `days_with_ga4` — counts already-passed days, not future behavior.

**Label:** `is_declining` = 1 if April 2026 impressions < March 2026
impressions, else 0 — a future-window outcome, not a reproduction of any
current-window product bucket.

**Baseline:** hand-written rule — flag if `content_age_days >= 180` AND
`impressions_march >= 500`, scored by traffic volume among qualifying pages.

**Models:** Logistic Regression and Random Forest, both `class_weight="balanced"`,
`random_state=42` fixed throughout for reproducibility.

**Validation design:** client-grouped split (`GroupShuffleSplit`, 75/25) —
no client's pages appear in both train and test.

**Leakage checks performed:** confirmed no product decision flags and no
`trend_direction`/`trend_pct` (label-derived fields) in any feature;
confirmed no preprocessing fit on the full dataset before splitting; and
directly demonstrated the cost of skipping this discipline — a naive random
split on the same model inflated Precision@50 from 0.700 to 0.900 (see
Results and `w06_validation_audit.ipynb`).

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [5]:
import json, pandas as pd

# Loads the committed receipts from w07_action_playbook.ipynb / w06_validation_audit.ipynb
with open("/outputs/playbook_metrics.json") as f:
    metrics = json.load(f)

results = pd.DataFrame([
    {"method": "Baseline rule (stale + visible)", "precision_at_50": metrics["baseline_precision_at_50"]},
    {"method": "Base rate (random guessing)", "precision_at_50": metrics["base_rate"]},
    {"method": "Logistic regression", "precision_at_50": 0.660},
    {"method": "Random forest (grouped split, honest)", "precision_at_50": metrics["precision_at_50_grouped_split"]},
    {"method": "Random forest (random split, leaky)", "precision_at_50": metrics["precision_at_50_random_split"]},
])
print(results.to_string(index=False))
print(f"\nClients scored: {metrics['n_clients']}  |  Pages scored: {metrics['n_pages_scored']}  |  "
      f"Validation month: {metrics['validation_month']}  |  Split: {metrics['split_type']}")

FileNotFoundError: [Errno 2] No such file or directory: '/outputs/playbook_metrics.json'

**Honest reading:** the baseline rule (0.500) scored *below* the base rate
(0.578) on this split — it missed more true positives than it caught. Both
learned models beat both the rule and the base rate under the honest,
client-grouped split; random forest reached 0.700. The same random forest
scored 0.900 under a naive random split — a 0.200 gap that is client-
identifiable leakage, not a real performance difference.

**Feature importance (random forest):** `avg_position` (0.366) >
`content_age_days` (0.287) > `impressions_march` (0.284) >> `days_with_ga4`
(0.063).

## 5. Limitations

*What this work cannot claim.*

- **Decision-support, not causal** — a flagged page is a review candidate,
  not proof that refreshing it will recover traffic. No experiment or
  matched control was run.
- **Single snapshot** — one month-to-month window (March→April 2026);
  seasonal effects have not been tested against other months.
- **Unbalanced panel** — client history depth varies substantially; results
  may not generalize evenly across all client types.
- **Score formula limitation** — within the baseline's qualifying group,
  ranking is driven by raw traffic alone, not position.
- **Residual risk on `content_age_days`** — may partly encode a training-
  split-specific pattern; three concrete false-positive cases in
  development shared this profile (young, well-positioned,
  high-confidence-but-wrong), so this feature's contribution should not be
  treated as fully generalizable without further validation.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:
recommendations = pd.DataFrame([
    {"reason_code": "stale_high_value", "action": "Review for refresh",
     "condition": "age >= 365 days AND impressions >= 500"},
    {"reason_code": "aging_visible_page", "action": "Review for refresh",
     "condition": "age >= 180 days AND impressions >= 500"},
    {"reason_code": "visible_low_rank", "action": "Review for CTR/intent fix",
     "condition": "impressions >= 500 AND avg_position > 10"},
])
print(recommendations.to_string(index=False))

**Human review required before any action:** check whether the page was
recently updated outside the logged staleness field; whether a sibling page
may be absorbing the same demand (consolidation, not decline); and whether
the reason code matches what a reviewer sees on the page.

**Never automate:** no page should be auto-refreshed or auto-published from
this output alone; product decision flags must never be fed back in as
features; no external claim should use causal language.

**Monitoring triggers:** a material shift in client mix, a Precision@50 drop
on a new month vs. the 0.700 baseline, or a sharp change in how many pages
clear the thresholds each month.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
import matplotlib.pyplot as plt
import os

os.makedirs("work/figures", exist_ok=True)

fig, ax = plt.subplots(figsize=(6, 4))
methods = ["Baseline\nrule", "Base\nrate", "Logistic\nregression", "Random forest\n(grouped, honest)", "Random forest\n(random, leaky)"]
scores = [metrics["baseline_precision_at_50"], metrics["base_rate"], 0.660,
          metrics["precision_at_50_grouped_split"], metrics["precision_at_50_random_split"]]
colors = ["#5A5A5A", "#A0A0A0", "#0891B2", "#087A96", "#D97757"]
ax.bar(methods, scores, color=colors)
ax.set_ylabel("Precision@50")
ax.set_title("Model vs. baseline, honest split vs. leaky split")
ax.set_ylim(0, 1)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig("work/figures/precision_at_50_comparison.png", dpi=150)
plt.show()

print("Saved: work/figures/precision_at_50_comparison.png")

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

---
# ML-12 — Tell the Story

The paper's Abstract and Introduction already tie this work to the real
FlyRank content-review problem (see Section 1 above and the deployed paper) —
that case-study framing lives inside the paper itself, not a separate file.
Below is the 5-minute demo outline and the two shareable cuts.

## 5-minute demo outline

*Ready for the Week-8 showcase, if presenting.*

**1. Question (30 sec):** FlyRank content teams have too many pages and too
little review time. Which pages should they look at first?

**2. Method (90 sec):** Built a scoring model on FlyRank's real 79M-row
search warehouse — observable pre-decision signals only (impressions,
position, age, analytics availability), validated with a client-grouped
split so the model can't memorize client-specific shortcuts.

**3. One chart (60 sec):** `work/figures/precision_at_50_comparison.png` —
five bars: baseline (0.500), base rate (0.578), logistic regression (0.660),
random forest honest (0.700), random forest leaky (0.900). Point at the gap
between the last two bars — that's the whole methodology lesson in one image.

**4. One honest result (60 sec):** The hand-written baseline actually scored
*below* random guessing. The model fixed that — but only once validated
honestly; the naive split would have shipped a model that looked 20 points
better than it really was.

**5. One recommendation (60 sec):** Ranked review queue with reason codes
(`stale_high_value`, `aging_visible_page`, `visible_low_rank`) — always
human-reviewed, never auto-actioned.

## Two shareable cuts

**Short social post (methodology-focused):**

> Validated a content-scoring ML model on FlyRank's real 79M-row search
> warehouse. The interesting finding wasn't the winning model — it was
> catching my own model cheating. A naive random train/test split showed
> Precision@50 of 0.900. Client-grouped validation (no client's pages in
> both train and test) dropped that to the honest number: 0.700. That
> 0.200 gap was data leakage — the model partly memorizing client-specific
> patterns instead of learning signal that generalizes. Lesson: validate
> the way you'll actually deploy, not the way that gives the prettiest
> score.

**3-sentence employer-facing summary:**

> I built a machine learning model that ranks which content pages a team
> should review first, using FlyRank's real 79-million-row search
> performance warehouse. Under honest, client-grouped validation, the
> model correctly flagged 70% of its top 50 recommended pages as genuinely
> worth reviewing — beating both a hand-written baseline rule and random
> chance. The project also demonstrates a concrete leakage-detection
> process: a naive validation split had overstated the same model's
> performance by 20 points, a mistake I caught and fixed before trusting
> the result.